# ResNet 2-class vs 7-class 비교 노트북
- `.py` 학습 스크립트를 노트북에서 실행
- history json 로드 후 Loss/Accuracy/LR 동일 포맷 비교

In [1]:
# 0) 커널/패키지 체크
import sys, subprocess, importlib.util
from pathlib import Path

BASE = Path(r"c:\\Users\\ldy34\\Desktop\\Face")
ML = BASE / "ML"

print("Current Python:", sys.executable)
if "Face\\.venv\\Scripts\\python.exe" not in sys.executable.replace('/', '\\'):
    raise RuntimeError("Kernel을 프로젝트 .venv (c:/Users/ldy34/Desktop/Face/.venv/Scripts/python.exe)로 바꿔주세요.")

required = ["matplotlib", "cv2", "mediapipe", "torch", "sklearn", "tqdm"]
missing = []
for m in required:
    if importlib.util.find_spec(m) is None:
        missing.append(m)

if missing:
    print("Installing missing packages:", missing)
    pkg_map = {"cv2": "opencv-python", "sklearn": "scikit-learn", "mediapipe": "mediapipe==0.10.14"}
    pkgs = [pkg_map.get(x, x) for x in missing]
    subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])
    print("설치 완료. 커널 재시작 후 다시 실행하세요.")
else:
    print("필수 패키지 준비 완료")

Current Python: c:\Users\ldy34\Desktop\Face\.venv\Scripts\python.exe
필수 패키지 준비 완료


In [2]:
# 1) 경로 설정
from pathlib import Path

BASE = Path(r"c:\\Users\\ldy34\\Desktop\\Face")
ML = BASE / "ML"
VENV_PY = BASE / ".venv" / "Scripts" / "python.exe"

TRAIN_7_PY = ML / "train_resnet_7class_mp.py"
TRAIN_2_PY = ML / "train_resnet_2class_mp.py"  # 있으면 사용

BIN_MODEL = ML / "emotion_resnet_best.pth"
BIN_HIST = ML / "emotion_resnet_best_history.json"

MC_MODEL = ML / "emotion_resnet_7class_best.pth"
MC_HIST = ML / "emotion_resnet_7class_best_history.json"

print("7-class trainer:", TRAIN_7_PY, TRAIN_7_PY.exists())
print("2-class trainer:", TRAIN_2_PY, TRAIN_2_PY.exists())
print("2-class history:", BIN_HIST.exists())
print("7-class history:", MC_HIST.exists())

7-class trainer: c:\Users\ldy34\Desktop\Face\ML\train_resnet_7class_mp.py True
2-class trainer: c:\Users\ldy34\Desktop\Face\ML\train_resnet_2class_mp.py False
2-class history: False
7-class history: True


In [ ]:
# 2) 7-class 학습 실행
import subprocess

cmd7 = [
    str(VENV_PY), str(TRAIN_7_PY),
    "--base-dir", str(BASE / "video"),
    "--epochs", "200",
    "--batch-size", "256",
    "--lr", "0.001",
    "--test-size", "0.15",
    "--max-per-zip", "5000",
    "--workers", "8",
    "--out", str(MC_MODEL),
]

print("RUN:", " ".join(cmd7))
res7 = subprocess.run(cmd7, cwd=str(BASE), check=True)
print("DONE:", res7)

In [ ]:
# 3) 2-class history 파일 체크
if not BIN_HIST.exists():
    raise FileNotFoundError(
        f"{BIN_HIST} 가 없습니다.\n"
        "2클래스 노트북에서 history를 저장하거나 train_resnet_2class_mp.py를 만든 뒤 생성해야 비교가 가능합니다."
    )

if not MC_HIST.exists():
    raise FileNotFoundError(f"{MC_HIST} 가 없습니다. 7-class 학습 셀을 먼저 실행하세요.")

print("history files ready")

In [ ]:
# 4) 결과 시각화 (동일 포맷)
import json
import matplotlib.pyplot as plt

with open(BIN_HIST, "r", encoding="utf-8") as f:
    history_bin = json.load(f)

with open(MC_HIST, "r", encoding="utf-8") as f:
    history_7 = json.load(f)

plt.figure(figsize=(18, 10))

# 2-class
plt.subplot(2, 3, 1)
plt.plot(history_bin['train_loss'], label='Train')
plt.plot(history_bin['test_loss'], label='Val')
plt.title('2-Class Loss')
plt.legend(); plt.grid(True)

plt.subplot(2, 3, 2)
plt.plot(history_bin['train_acc'], label='Train')
plt.plot(history_bin['test_acc'], label='Val')
plt.title('2-Class Accuracy')
plt.legend(); plt.grid(True)

plt.subplot(2, 3, 3)
plt.plot(history_bin['lr'], color='red')
plt.title('2-Class Learning Rate')
plt.grid(True)

# 7-class
plt.subplot(2, 3, 4)
plt.plot(history_7['train_loss'], label='Train')
plt.plot(history_7['test_loss'], label='Val')
plt.title('7-Class Loss')
plt.legend(); plt.grid(True)

plt.subplot(2, 3, 5)
plt.plot(history_7['train_acc'], label='Train')
plt.plot(history_7['test_acc'], label='Val')
plt.title('7-Class Accuracy')
plt.legend(); plt.grid(True)

plt.subplot(2, 3, 6)
plt.plot(history_7['lr'], color='red')
plt.title('7-Class Learning Rate')
plt.grid(True)

plt.tight_layout()
plt.show()